# Session 09: Reading Files and Handling Errors

Data Science with Python, Level 1, Stage 1: Python Foundations

## Learning Objectives

By the end of this session you will be able to:

1. Write and read text files with `open` and `with`.
2. Process a file line by line.
3. Read and write CSV files.
4. Read error messages and name common errors.
5. Handle errors with `try` and `except`.
6. Raise your own errors for invalid data.

## Where We Are

Until now, all our data was typed into the notebook, and every result disappeared when the notebook closed. Real data lives in **files**: the CSV exports of a sales system, the logs of a website, the answers to a survey. Programs read those files, and write their results back to new files.

Real files are also messy. A single line with a letter where a number should be can crash a program that has processed thousands of lines correctly. Today you will learn to **handle errors**, so your program can report the problem and keep going.

In [ ]:
values = ["12.5", "8", "seven", "4.25"]
total = 0
for v in values:
    total += float(v)

## 1. Writing and Reading Text Files

`open(name, mode)` connects your program to a file. The mode `"w"` **writes** a new file (replacing any old one), `"a"` **appends** to the end, and `"r"` **reads**, which is the default.

Always open files with a `with` block. When the block ends, Python closes the file automatically, even if an error happens inside it.

In Colab, files are saved in the notebook's temporary folder. Click the folder icon on the left to see them. They are deleted when the runtime restarts, so download anything you want to keep.

In [ ]:
with open("notes.txt", "w") as f:
    f.write("Session 09: Files and Errors\n")
    f.write("Files keep data after the program ends.\n")

with open("notes.txt") as f:
    text = f.read()

print(text)
print(len(text), "characters")

In [ ]:
with open("notes.txt", "a") as f:
    f.write("Appending adds lines to the end.\n")

with open("notes.txt") as f:
    for number, line in enumerate(f, start=1):
        print(number, line.strip())

| Mode | Meaning | If the file exists | If it does not exist |
| --- | --- | --- | --- |
| `"r"` | Read (the default) | Reads it | `FileNotFoundError` |
| `"w"` | Write | Replaces everything | Creates it |
| `"a"` | Append | Adds to the end | Creates it |

## 2. CSV Files

A **CSV** file stores a table as text: one row per line, with the values separated by commas. The first line usually holds the column names. Every spreadsheet program can save and open CSV files, and pandas reads them in one line, as you will see in Session 16.

Here we write a small CSV file and read it with what you already know: `split(",")`.

In [ ]:
rows = [
    "item,category,price,sold",
    "Latte,Coffee,3.5,128",
    "Croissant,Pastry,2.8,64",
    "Green Tea,Tea,2.5,41",
]
with open("menu.csv", "w") as f:
    f.write("\n".join(rows) + "\n")

In [ ]:
with open("menu.csv") as f:
    header = f.readline().strip().split(",")
    for line in f:
        item, category, price, sold = line.strip().split(",")
        print(f"{item:<10} revenue {float(price) * int(sold):.2f}")

print(header)

In [ ]:
import csv

with open("menu.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["item"], row["price"])

## 3. Reading Error Messages

When Python cannot run a line, it stops and shows an **error message**, also called a traceback. Read it from the **bottom**: the last line names the type of error and gives a short description. The lines above show where it happened.

You have already met several errors. Here are the ones you will see most in data work.

| Error | Typical cause | Example |
| --- | --- | --- |
| `NameError` | A name that was never defined, often a typo | `print(totl)` |
| `TypeError` | An operation on the wrong type | `"5" + 3` |
| `ValueError` | The right type but an unusable value | `int("seven")` |
| `IndexError` | A list index that does not exist | `[1, 2][5]` |
| `KeyError` | A dictionary key that does not exist | `{"a": 1}["b"]` |
| `ZeroDivisionError` | Dividing by zero | `10 / 0` |
| `FileNotFoundError` | A wrong file name or folder | `open("data.cvs")` |

In [ ]:
with open("sales_2025.csv") as f:
    print(f.read())

## 4. Handling Errors with try and except

Code that might fail goes inside a `try` block. If an error happens, Python jumps to the matching `except` block instead of stopping. Always name the error you expect, such as `except ValueError:`, so that other, unexpected errors are still reported.

In [ ]:
values = ["12.5", "8", "seven", "4.25"]
total = 0
for v in values:
    try:
        total += float(v)
    except ValueError:
        print(f"Skipping bad value: {v!r}")

print("Total:", total)

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    else:
        return round(result, 2)
    finally:
        print("Division attempted:", a, "/", b)

print(safe_divide(10, 4))
print(safe_divide(10, 0))

In [ ]:
def read_numbers(filename):
    try:
        with open(filename) as f:
            return [float(line) for line in f if line.strip()]
    except FileNotFoundError:
        print(f"File {filename} not found, using an empty list")
        return []

print(read_numbers("missing.txt"))

## 5. Raising Your Own Errors

Sometimes the data is valid Python but makes no sense: a negative price, an age of 230, an empty list where values are needed. Use `raise` to stop with a clear message. It is much better to fail loudly with a clear reason than to continue and produce a wrong answer.

In [ ]:
def apply_discount(price, rate):
    if not 0 <= rate <= 1:
        raise ValueError(f"rate must be between 0 and 1, got {rate}")
    return price * (1 - rate)

print(apply_discount(100, 0.2))
print(apply_discount(100, 20))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Data lost | Opening an existing file with mode `"w"` | `"w"` replaces the file; use `"a"` to add to it. |
| TypeError | Doing maths with values read from a file | Everything read from a text file is a string; convert it. |
| Hidden bugs | Using a bare `except:` that catches everything | Name the error: `except ValueError:`. |
| FileNotFoundError | A typo in the file name or extension | Check the Files panel; names are case sensitive. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Diary

Write three lines about your day to a file called `diary.txt`. Then append a fourth line, and print the whole file with line numbers.

In [ ]:
# Write your code below

### In-Class Exercise 2: Safe Input

Write `to_number(text)` that returns the text as a float, or `None` if it cannot be converted. Use it to add up only the valid values in `["3", "4.5", "abc", "", "10"]`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Average from a File

Write the scores `72, 88, 95, 64, 81` to a file `scores.txt`, one per line. Read them back, and print their count and average.

In [ ]:
# Write your code below

### In-Class Exercise 4: Validate an Age

Write `check_age(age)` that raises a `ValueError` with a clear message if the age is below 0 or above 120, and returns the age otherwise. Call it inside `try` for the ages 25, -3 and 230, and print the error message for the invalid ones.

In [ ]:
# Write your code below

## Summary

- `with open(name, mode) as f:` reads or writes a file safely.
- `"w"` replaces, `"a"` appends, `"r"` reads.
- Values read from text files are strings; convert them.
- Read error messages from the bottom line up.
- `try` and `except ValueError:` handle expected errors.
- `raise` stops early with a clear message.

## What's Next

Python comes with hundreds of ready-made modules, and thousands more can be installed. Next session you will import them, use `math`, `random`, `datetime` and `statistics`, and write a module of your own.